In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import cv2

from pathlib import Path
import pickle

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    import spotiphy

In [2]:
def run_spotiphy(
        adata_visium: sc.AnnData, adata_sc: sc.AnnData, cell_type_key: str, save_dir: str,
        image: np.ndarray = None, spot_shape: str = None, spot_radius: float = None, pixel_size: float = None) -> None:
    """Run Spotiphy.

    Args:
        adata_visium (sc.AnnData): AnnData object of Visium.
        adata_sc (sc.AnnData): AnnData object of scRNA-seq.
        cell_type_key (str): Key of cell type annotation in `adata_sc.obs`.
        save_dir (str): Directory to save the results.

    Returns:
        None.

    """
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    adata_visium_orig = adata_visium.copy()
    adata_sc_orig = adata_sc.copy()

    # preprocess
    print('==== Preprocessing ====')
    cell_type_list = sorted(list(adata_sc.obs[cell_type_key].unique().astype(str)))
    adata_sc, adata_visium = spotiphy.initialization(adata_sc, adata_visium)

    # select marker genes
    print('==== Selecting marker genes ====')
    marker_genes_dict = spotiphy.sc_reference.marker_selection(
        adata_sc,
        key_type=cell_type_key,
        return_dict=True,
        n_select=50,
        threshold_p=0.1,
        threshold_fold=1.5,
        q=0.15,
    )
    marker_genes = []
    marker_genes_label = []
    for type_ in cell_type_list:
        marker_genes.extend(marker_genes_dict[type_])
        marker_genes_label.extend([type_] * len(marker_genes_dict[type_]))

    # save
    pd.DataFrame({'gene': marker_genes, 'label': marker_genes_label}).to_csv('%s/marker_genes.csv' % save_dir, sep=',', index=False, header=True)

    # filter adata with marker genes
    adata_sc_marker = adata_sc[:, marker_genes].copy()
    adata_visium_marker = adata_visium[:, marker_genes].copy()

    # construct_sc_ref
    sc_ref = spotiphy.construct_sc_ref(adata_sc_marker, key_type=cell_type_key)

    # estimate cell proportion
    print('==== Estimating cell proportion ====')
    X = np.array(adata_visium_marker.X)
    cell_proportion = spotiphy.deconvolution.estimation_proportion(
        X,
        adata_sc_marker,
        sc_ref,
        cell_type_list,
        cell_type_key,
        n_epoch=8000,
        plot=False,
        batch_prior=1,
        device='cuda',
    )

    # save
    pd.DataFrame(
        cell_proportion,
        index=adata_visium.obs_names,
        columns=cell_type_list
    ).to_csv('%s/cell_proportion.csv' % save_dir, sep=',', index=True, header=True)

    # nuclear segment
    print('==== Segmenting nuclei ====')
    segmentation = spotiphy.segmentation.Segmentation(
        image,
        adata_visium.obsm['spatial'],
        spot_radius=spot_radius / pixel_size,
        spot_shape=spot_shape,
        n_tiles=(2, 2, 1),
        out_dir='%s/nuclear_segmentation/' % save_dir,
        enhancement=True,
    )
    segmentation.segment_nucleus(save=True)

    # decompose all genes in the spatial transcriptomics data
    print('==== Decomposing gene expression ====')
    adata_visium_decomposed = spotiphy.deconvolution.decomposition(
        adata_visium_orig,
        adata_sc_orig,
        cell_type_key,
        cell_proportion,
        save=True,
        out_dir=save_dir,
        verbose=1,
        spot_location=adata_visium_orig.obsm['spatial'],
        filtering_gene=True,
        n_cell=segmentation.n_cell_df['cell_count'].values,
        filename='adata_visium_decomposed.h5ad',
    )

    # pseudo single-cell resolution image
    print('==== Assigning cell types ====')
    cell_number = spotiphy.deconvolution.proportion_to_count(
        cell_proportion,
        segmentation.n_cell_df['cell_count'].values,
        multiple_spots=True
    )
    segmentation.nucleus_df = spotiphy.deconvolution.assign_type_spot(
        segmentation.nucleus_df,
        segmentation.n_cell_df,
        cell_number,
        cell_type_list
    )
    segmentation.nucleus_df, cell_proportion_smooth = (
        spotiphy.deconvolution.assign_type_out(
            segmentation.nucleus_df,
            cell_proportion,
            segmentation.spot_center,
            cell_type_list,
            max_distance=100 / pixel_size,
            band_width=150,
        )
    )
    with open('%s/nuclear_segmentation/segmentation.pkl' % save_dir, 'wb') as file:
        pickle.dump(segmentation, file)

    return

In [3]:
visium_adata_file = '/data1/hounaiqiao/wzr/Simulated_Xenium/luca/w100/simulated_data_with_interval.h5ad'
sc_adata_file = '/data1/hounaiqiao/yy/NucleiSeg/benchmark_review/interval/data/luca/sc_reference_12.h5ad'
image_file = '/data1/hounaiqiao/wzr/Simulated_Xenium/luca/HE_aligned/wholeslide/align_he.png'
save_dir = '/data1/hounaiqiao/yy/NucleiSeg/benchmark_review/interval/results/luca_w100/spotiphy/'

In [4]:
adata_visium = sc.read_h5ad(visium_adata_file)
adata_sc = sc.read_h5ad(sc_adata_file)
image = cv2.cvtColor(cv2.imread(image_file), cv2.COLOR_BGR2RGB)

In [5]:
run_spotiphy(adata_visium, adata_sc, 'label', save_dir, image, 'square', 100, 0.2125)

==== Preprocessing ====
==== Selecting marker genes ====


12it [00:00, 170.70it/s]


==== Estimating cell proportion ====


100%|██████████| 8000/8000 [03:29<00:00, 38.20it/s]


==== Segmenting nuclei ====
Suppress the output of tensorflow prediction for tensorflow version 2.16.2>=2.9.0.
Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.


100%|██████████| 4/4 [01:05<00:00, 16.26s/it]


==== Decomposing gene expression ====
Prepared proportion data. Time use 0.03
Initialized scRNA and ST data. Time use 69.37


12it [00:00, 52.01it/s]


Processed scRNA and ST data. Time use 2.18
Decomposition complete. Time use 0.01
Constructed ST decomposition data file. Time use 1.26
Saved file to output folder. Time use 2.70
==== Assigning cell types ====


deconvolution.py (1206): invalid value encountered in divide
